# Generative distributions for the theorem illustration

This notebook sets up the two distributions that will later be used to compare the volumes of HDR and optimal-transport regions:

- the uniform distribution on the solid Euclidean ball $B_d(0, 1)$;
- a centered Gaussian with covariance
  $$\Sigma_{d,k} = \operatorname{diag}\left(k^{d - 1/d}, k^{-1/d}, \ldots, k^{-1/d}\right).$$

The notebook also constructs Gaussian HDRs by radially transporting nested source balls.

In [ ]:
from dataclasses import asdict
from itertools import product
from pathlib import Path
import sys

import pandas as pd
import torch
from tqdm.auto import tqdm


def find_experiment_directory(start=Path.cwd()):
    for candidate in (start.resolve(), *start.resolve().parents):
        direct = candidate / "utils.py"
        nested = candidate / "notebooks/illustrating_the_theorem/utils.py"
        if direct.is_file():
            return direct.parent
        if nested.is_file():
            return nested.parent
    raise FileNotFoundError("Could not locate illustrating_the_theorem/utils.py.")


EXPERIMENT_DIRECTORY = find_experiment_directory()
if str(EXPERIMENT_DIRECTORY) not in sys.path:
    sys.path.insert(0, str(EXPERIMENT_DIRECTORY))

from utils import (
    empirical_ot_transport_cost,
    empirical_ot_transport_pairs,
    estimate_ot_induced_region_volume,
    gaussian_covariance_diagonal,
    gaussian_hdr_log_volume,
    log_ball_volume,
    sample_empirical_ot,
    sample_gaussian,
    sample_gaussian_hdr,
    sample_uniform_ball,
    run_log_volume_ratio_experiment,
    transport_uniform_ball_to_gaussian,
)

torch.set_default_dtype(torch.float64)

## Experiment parameters

In [ ]:
DIMENSION = 2
K = 4.0
RADIUS = 1.0
SIGNIFICANCE_LEVEL = 0.1
COVERAGE_MASS = 1.0 - SIGNIFICANCE_LEVEL
NUMBER_OF_SAMPLES = 100_000
NUMBER_OF_OT_POINTS = 1_000
NUMBER_OF_VOLUME_SAMPLES = 100_000
SEED = 42
OT_BACKEND = "sinkhorn"  # dense_exact, lazy_exact, or sinkhorn
SINKHORN_EPSILON = 0.1
SINKHORN_COMPUTE_BACKEND = "cuda"  # cpu, cuda, or geomloss
OT_MAXIMUM_ITERATIONS = 10_000 if OT_BACKEND == "sinkhorn" else 1_000_000
OT_TOLERANCE = 1e-9
OT_BATCH_SIZE = 256

(
    DIMENSION,
    K,
    RADIUS,
    SIGNIFICANCE_LEVEL,
    COVERAGE_MASS,
    NUMBER_OF_SAMPLES,
    NUMBER_OF_OT_POINTS,
    NUMBER_OF_VOLUME_SAMPLES,
    OT_BACKEND,
    SINKHORN_COMPUTE_BACKEND,
    torch.cuda.is_available(),
)

(2, 4.0, 1.0, 0.1, 0.9, 100000, 1000, 100000, 'lazy_exact')

## Generate samples

In [ ]:
uniform_generator = torch.Generator().manual_seed(SEED)
gaussian_generator = torch.Generator().manual_seed(SEED + 1)

uniform_samples = sample_uniform_ball(
    number_of_samples=NUMBER_OF_SAMPLES,
    dimension=DIMENSION,
    radius=RADIUS,
    generator=uniform_generator,
)
gaussian_samples = sample_gaussian(
    number_of_samples=NUMBER_OF_SAMPLES,
    dimension=DIMENSION,
    k=K,
    generator=gaussian_generator,
)

uniform_samples.shape, gaussian_samples.shape

(torch.Size([100000, 2]), torch.Size([100000, 2]))

## Generate a Gaussian HDR

A source ball of radius $q^{1/d}$ contains mass $q$. The radial map sends $r$ to $F_{\chi_d}^{-1}(r^d)$, after which multiplication by $\Sigma_{d,k}^{1/2}$ produces the target Gaussian HDR.

In [ ]:
hdr_generator = torch.Generator().manual_seed(SEED + 2)
hdr_samples = sample_gaussian_hdr(
    number_of_samples=NUMBER_OF_SAMPLES,
    dimension=DIMENSION,
    k=K,
    coverage_mass=COVERAGE_MASS,
    generator=hdr_generator,
)

# The pointwise map is also available for arbitrary points in the unit ball.
transported_uniform_samples = transport_uniform_ball_to_gaussian(
    uniform_samples, k=K
)

hdr_samples.shape, transported_uniform_samples.shape

(torch.Size([100000, 2]), torch.Size([100000, 2]))

## Solve empirical OT with a selectable backend

Set `OT_BACKEND` to `"dense_exact"`, `"lazy_exact"`, or `"sinkhorn"`. When Sinkhorn is selected, set `SINKHORN_COMPUTE_BACKEND` explicitly to `"cpu"` (POT's default lazy solver), `"cuda"` (POT's GeomLoss/PyKeOps solver forced onto CUDA), or `"geomloss"` (the same POT integration with automatic device selection). The configuration cell currently selects `"cuda"`; `torch.cuda.is_available()` is displayed alongside the selected values. The exact solvers produce one-to-one matches; Sinkhorn produces a smooth barycentric image for every source point. The later volume calculation accepts all three result types without converting Sinkhorn into a permutation.

In [ ]:
ot_source_generator = torch.Generator().manual_seed(SEED + 3)
ot_target_generator = torch.Generator().manual_seed(SEED + 4)

ot_result = sample_empirical_ot(
    number_of_points=NUMBER_OF_OT_POINTS,
    dimension=DIMENSION,
    k=K,
    backend=OT_BACKEND,
    epsilon=SINKHORN_EPSILON,
    sinkhorn_compute_backend=SINKHORN_COMPUTE_BACKEND,
    source_generator=ot_source_generator,
    target_generator=ot_target_generator,
    maximum_iterations=OT_MAXIMUM_ITERATIONS,
    tolerance=OT_TOLERANCE,
    batch_size=OT_BATCH_SIZE,
)

ot_source_points, transported_ot_points = empirical_ot_transport_pairs(
    ot_result
)
ot_source_region_radius = COVERAGE_MASS ** (1.0 / DIMENSION)
ot_source_region_mask = (
    ot_source_points.norm(dim=-1) <= ot_source_region_radius
)
ot_induced_region_points = transported_ot_points[ot_source_region_mask]

print(f"OT backend: {OT_BACKEND}")
if OT_BACKEND == "sinkhorn":
    print(f"Sinkhorn compute backend: {SINKHORN_COMPUTE_BACKEND}")
print(f"Computed {len(ot_source_points)} transported source points.")
print(
    "Mean squared transport cost: "
    f"{empirical_ot_transport_cost(ot_result):.6f}"
)
print(
    f"Points induced by the {COVERAGE_MASS:.0%} source region: "
    f"{len(ot_induced_region_points)}"
)

OT backend: lazy_exact
Computed 1000 transported source points.
Mean squared transport cost: 5.217065
Points induced by the 90% source region: 893


## Estimate the OT-induced region volume

The transported points associated with the latent ball of mass $1-\alpha$ define a coordinate-wise bounding box. Monte Carlo points drawn uniformly from this box are pulled back through the nearest transported atom. Exact backends use matched targets, while Sinkhorn uses its barycentric map.

In [ ]:
volume_generator = torch.Generator().manual_seed(SEED + 5)
ot_volume_estimate = estimate_ot_induced_region_volume(
    matching=ot_result,
    significance_level=SIGNIFICANCE_LEVEL,
    number_of_monte_carlo_samples=NUMBER_OF_VOLUME_SAMPLES,
    generator=volume_generator,
)

print(f"Bounding-box lower limits: {ot_volume_estimate.lower_bounds}")
print(f"Bounding-box upper limits: {ot_volume_estimate.upper_bounds}")
print(f"Log bounding-box volume: {ot_volume_estimate.log_bounding_box_volume:.6f}")
print(f"Log Monte Carlo inclusion fraction: {ot_volume_estimate.log_inclusion_fraction:.6f}")
print(
    f"Log OT-induced volume: {ot_volume_estimate.log_volume:.6f} "
    f"+/- {ot_volume_estimate.log_volume_mc_standard_error:.6f} "
    "(delta-method MC standard error)"
)

Bounding-box lower limits: tensor([-6.2428, -1.3978])
Bounding-box upper limits: tensor([6.3486, 2.1485])
Log bounding-box volume: 3.798908
Log Monte Carlo inclusion fraction: -0.408164
Log OT-induced volume: 3.390744 +/- 0.002245 (delta-method MC standard error)


## Basic diagnostics

The empirical moments below provide a quick check that the generators agree with the requested distributions.

In [ ]:
target_covariance_diagonal = gaussian_covariance_diagonal(DIMENSION, K)
empirical_covariance_diagonal = gaussian_samples.var(dim=0, correction=0)
maximum_uniform_radius = uniform_samples.norm(dim=-1).max()
transported_covariance_diagonal = transported_uniform_samples.var(
    dim=0, correction=0
)
hdr_mahalanobis_radii = (
    hdr_samples.square() / target_covariance_diagonal
).sum(dim=-1).sqrt()
exact_hdr_log_volume = gaussian_hdr_log_volume(
    dimension=DIMENSION,
    k=K,
    significance_level=SIGNIFICANCE_LEVEL,
)

print(f"Log-volume of B_{DIMENSION}(0, {RADIUS:g}): {log_ball_volume(DIMENSION, RADIUS):.6f}")
print(f"Largest sampled uniform radius: {maximum_uniform_radius:.6f}")
print(f"Uniform empirical mean: {uniform_samples.mean(dim=0)}")
print(f"Gaussian empirical mean: {gaussian_samples.mean(dim=0)}")
print(f"Target Gaussian covariance diagonal: {target_covariance_diagonal}")
print(f"Direct Gaussian covariance diagonal: {empirical_covariance_diagonal}")
print(f"Transported covariance diagonal: {transported_covariance_diagonal}")
print(f"Largest HDR Mahalanobis radius: {hdr_mahalanobis_radii.max():.6f}")
print(f"Exact {COVERAGE_MASS:.0%} Gaussian HDR log-volume: {exact_hdr_log_volume:.6f}")

Log-volume of B_2(0, 1): 1.144730
Largest sampled uniform radius: 1.000000
Uniform empirical mean: tensor([0.0003, 0.0006])
Gaussian empirical mean: tensor([ 0.0074, -0.0013])
Target Gaussian covariance diagonal: tensor([8.0000, 0.5000])
Direct Gaussian covariance diagonal: tensor([7.9811, 0.4963])
Transported covariance diagonal: tensor([7.9867, 0.4993])
Largest HDR Mahalanobis radius: 2.145897
Exact 90% Gaussian HDR log-volume: 3.365057


## Final log-volume ratio

Report $\log\left(\operatorname{Vol}(\mathrm{HDR}^{f}_{1-\alpha}) / \operatorname{Vol}(\mathrm{CC}^{\hat{T}}_{1-\alpha})\right)$ directly as a difference of log-volumes.

In [ ]:
log_volume_ratio = exact_hdr_log_volume - ot_volume_estimate.log_volume
print(
    "log(Vol(HDR^f_{1-alpha}) / Vol(CC^{T_hat}_{1-alpha})) = "
    f"{log_volume_ratio:.6f} +/- "
    f"{ot_volume_estimate.log_volume_mc_standard_error:.6f} "
    "(MC standard error)"
)

log(Vol(HDR^f_{1-alpha}) / Vol(CC^{T_hat}_{1-alpha})) = -0.025688 +/- 0.002245 (MC standard error)


## Dimension and anisotropy sweep

Run seeds 0 through 9 for dimensions $2, 4, 8, 16, 32, 64$ and $k \in \{2, 4, 8, 16\}$. Every run uses 10,000 empirical atoms and 10,000 Monte Carlo samples for the induced-region log-volume. `SWEEP_OT_BACKEND` selects dense exact, lazy exact, or Sinkhorn transport. Results are checkpointed after every seed and an existing compatible checkpoint is resumed automatically.

> **Resource note:** `dense_exact` materializes dense cost and coupling matrices. `lazy_exact` evaluates costs on demand. Sinkhorn uses either POT's default lazy blocks or POT's online GeomLoss/PyKeOps integration; both produce the barycentric map without storing the full coupling.

In [ ]:
SWEEP_DIMENSIONS = [2, 4, 8, 16, 32, 64]
SWEEP_K_VALUES = [2]
SWEEP_SEEDS = list(range(10))
SWEEP_NUMBER_OF_OT_POINTS = 1000
SWEEP_NUMBER_OF_MONTE_CARLO_SAMPLES = 10_000
SWEEP_MONTE_CARLO_BATCH_SIZE = 10_000
SWEEP_OT_BACKEND = "sinkhorn"  # dense_exact, lazy_exact, or sinkhorn
SWEEP_SINKHORN_EPSILON = 1e-1
SWEEP_SINKHORN_COMPUTE_BACKEND = SINKHORN_COMPUTE_BACKEND
SWEEP_OT_MAXIMUM_ITERATIONS = OT_MAXIMUM_ITERATIONS
SWEEP_OT_TOLERANCE = OT_TOLERANCE
SWEEP_OT_BATCH_SIZE = OT_BATCH_SIZE
SWEEP_BACKEND_TAG = SWEEP_OT_BACKEND
if SWEEP_OT_BACKEND == "sinkhorn":
    SWEEP_BACKEND_TAG += (
        f"_eps_{SWEEP_SINKHORN_EPSILON:g}"
        f"_{SWEEP_SINKHORN_COMPUTE_BACKEND}"
    )
SWEEP_RESULTS_PATH = (
    EXPERIMENT_DIRECTORY
    / (
        f"4_only_log_volume_ratio_sweep_ot_{SWEEP_NUMBER_OF_OT_POINTS}"
        f"_mc_10000_{SWEEP_BACKEND_TAG}.csv"
    )
)
RESUME_SWEEP = True

SWEEP_RESULTS_PATH

PosixPath('/Users/vladimir.kondratyev/minimal_volume_conformal_prediction/notebooks/illustrating_the_theorem/4_only_log_volume_ratio_sweep_ot_1000_mc_10000_sinkhorn_eps_0.1.csv')

In [ ]:
def normalized_regularization(value):
    return None if value is None or pd.isna(value) else float(value)


def normalized_sinkhorn_compute_backend(record):
    if str(record.get("ot_backend", "dense_exact")) != "sinkhorn":
        return None
    value = record.get("sinkhorn_compute_backend")
    return "cpu" if value is None or pd.isna(value) else str(value)


def sweep_key(record):
    return (
        int(record["dimension"]),
        float(record["k"]),
        int(record["seed"]),
        float(record["significance_level"]),
        int(record["number_of_ot_points"]),
        int(record["number_of_monte_carlo_samples"]),
        str(record.get("ot_backend", "dense_exact")),
        normalized_regularization(record.get("regularization")),
        normalized_sinkhorn_compute_backend(record),
    )


if RESUME_SWEEP and SWEEP_RESULTS_PATH.is_file():
    sweep_records = pd.read_csv(SWEEP_RESULTS_PATH).to_dict(orient="records")
else:
    sweep_records = []
completed_sweep_keys = {sweep_key(record) for record in sweep_records}

dimension_k_pairs = list(product(SWEEP_DIMENSIONS, SWEEP_K_VALUES))
pair_progress = tqdm(
    dimension_k_pairs,
    desc="Dimension/k pairs",
    unit="pair",
)
for dimension, k in pair_progress:
    pair_progress.set_postfix(dimension=dimension, k=k)
    seeds_to_run = [
        seed
        for seed in SWEEP_SEEDS
        if (
            dimension,
            float(k),
            seed,
            SIGNIFICANCE_LEVEL,
            SWEEP_NUMBER_OF_OT_POINTS,
            SWEEP_NUMBER_OF_MONTE_CARLO_SAMPLES,
            SWEEP_OT_BACKEND,
            (
                SWEEP_SINKHORN_EPSILON
                if SWEEP_OT_BACKEND == "sinkhorn"
                else None
            ),
            (
                SWEEP_SINKHORN_COMPUTE_BACKEND
                if SWEEP_OT_BACKEND == "sinkhorn"
                else None
            ),
        )
        not in completed_sweep_keys
    ]
    seed_progress = tqdm(
        seeds_to_run,
        desc=f"d={dimension}, k={k}",
        unit="seed",
        leave=False,
    )
    for seed in seed_progress:
        result = run_log_volume_ratio_experiment(
            dimension=dimension,
            k=k,
            significance_level=SIGNIFICANCE_LEVEL,
            number_of_ot_points=SWEEP_NUMBER_OF_OT_POINTS,
            number_of_monte_carlo_samples=(
                SWEEP_NUMBER_OF_MONTE_CARLO_SAMPLES
            ),
            seed=seed,
            ot_backend=SWEEP_OT_BACKEND,
            sinkhorn_epsilon=SWEEP_SINKHORN_EPSILON,
            sinkhorn_compute_backend=(
                SWEEP_SINKHORN_COMPUTE_BACKEND
            ),
            ot_maximum_iterations=SWEEP_OT_MAXIMUM_ITERATIONS,
            ot_tolerance=SWEEP_OT_TOLERANCE,
            ot_batch_size=SWEEP_OT_BATCH_SIZE,
            monte_carlo_batch_size=SWEEP_MONTE_CARLO_BATCH_SIZE,
        )
        record = asdict(result)
        sweep_records.append(record)
        completed_sweep_keys.add(sweep_key(record))

        sweep_results = (
            pd.DataFrame.from_records(sweep_records)
            .drop_duplicates(
                subset=[
                    "dimension",
                    "k",
                    "seed",
                    "ot_backend",
                    "regularization",
                    "sinkhorn_compute_backend",
                ],
                keep="last",
            )
            .sort_values(["dimension", "k", "seed"])
            .reset_index(drop=True)
        )
        sweep_results.to_csv(SWEEP_RESULTS_PATH, index=False)

sweep_results = (
    pd.DataFrame.from_records(sweep_records)
    .drop_duplicates(
        subset=[
            "dimension",
            "k",
            "seed",
            "ot_backend",
            "regularization",
            "sinkhorn_compute_backend",
        ],
        keep="last",
    )
    .sort_values(["dimension", "k", "seed"])
    .reset_index(drop=True)
)
display(sweep_results)

Dimension/k pairs:   0%|          | 0/6 [00:00<?, ?pair/s]

d=2, k=2:   0%|          | 0/10 [00:00<?, ?seed/s]

d=4, k=2:   0%|          | 0/10 [00:00<?, ?seed/s]

d=8, k=2:   0%|          | 0/10 [00:00<?, ?seed/s]

In [ ]:
sweep_results = pd.read_csv(SWEEP_RESULTS_PATH)

In [ ]:
sweep_summary = (
    sweep_results.groupby(["dimension", "k"])["log_volume_ratio"]
    .agg(
        n_seeds="count",
        mean="mean",
        standard_deviation="std",
        median="median",
        minimum="min",
        maximum="max",
    )
    .reset_index()
)
display(sweep_summary)